### Importing Standard Libraries

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType
from pyspark.sql.functions import col
from delta import DeltaTable

### Paths and Variables

In [0]:
dbName = 'testDB'
tableTestName = f'{dbName}.testTable'
employeeTable = f'{dbName}.employee'
metastore_path = '/user/metastore'
employee_table_path = metastore_path + '/employee'
test_table_path = metastore_path + '/testtable'

In [0]:
display(dbutils.fs.ls(metastore_path))

### SQL operations

#### Creating database at location metastore
1. Creating database/schema using sql syntax

2. Creating database/schema using spark.sql()

3. Database commands
    * SHOW DATABASES/SCHEMAS
    * DESCRIBE DATABASES/SCHEMA {database_name}
    * DESCRIBE EXTENDED DATABASES/SCHEMA {database_name}
    
4. Table commands
    * SHOW TABLES 
    * DESCRIBE TABLE {table_name}
    * DESCRIBE EXTENDED TABLE {table_name}

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS testDB
COMMENT 'test Database for all tables'
LOCATION '/user/metastore'

In [0]:
# Creating using spark.sql()
spark.sql(f""" CREATE SCHEMA IF NOT EXISTS testDB
COMMENT 'test Database'
LOCATION '{metastore_path}'
""")

In [0]:
%sql
SHOW DATABASES
--SHOW SCHEMAS

In [0]:
%sql
DESCRIBE SCHEMA default

In [0]:
%sql
DESCRIBE DATABASE testDB
--DESCRIBE SCHEMA testDB

#### Create Table

1. CREATE OR REPLACE TABLE
2. CREATE TABLE IF NOT EXISTS
3. CREATE EXTERNAL TABLE OR REPLACE
4. CREATE EXTERNAL TABLE IF NOT EXISTS

5. CREATE TABLE LIKE
6. CREATE TABLE CLONE


##### time_table

In [0]:
dbutils.fs.ls('dbfs:/user/hive/warehouse/')
dbutils.fs.rm('dbfs:/user/hive/warehouse/time_table')

In [0]:
spark.sql("""CREATE OR REPLACE TABLE time_table (
id INT,
status STRING,
dates TIMESTAMP
)
USING DELTA;
""");

spark.sql("""
INSERT INTO time_table VALUES
  (1, 'Fail', '2022-01-01 00:00:00'),
  (2, 'Success', '2022-01-02 00:00:00'),
  (3, 'Fail', '2022-01-03 00:00:00'),
  (4, 'Success', '2022-01-04 00:00:00'),
  (5, 'Fail', '2022-01-05 00:00:00'),
  (6, 'Success', '2022-01-06 00:00:00'),
  (7, 'Fail', '2022-01-06 11:04:11'),
  (8, 'Fail', '2022-01-05 01:32:32'),
  (9, 'Fail', '2022-01-06 07:14:31'),
  (10, 'Success', '2022-01-06 23:56:59');
""")

In [0]:
%sql
SELECT * FROM time_table

In [0]:
# to filter for fail and max date 
spark.sql("""
SELECT *  FROM time_table WHERE 
DATE_FORMAT(dates, 'yyyy-MM-dd') = DATE_FORMAT((SELECT MAX(dates) FROM time_table),'yyyy-MM-dd') 
AND status = 'Fail'
""").display()

In [0]:
df_employee = spark.read.format("delta").load(employee_table_path)
df_employee.show()

In [0]:
df_test = spark.read.format("delta").load(test_table_path)
df_test.show()

In [0]:
%sql
CREATE OR REPLACE TABLE employee_delta (
  empno INT,
  ename STRING,
  designation STRING,
  manager INT,
  hire_date DATE,
  sal BIGINT,
  deptno INT,
  location STRING
) USING DELTA;

In [0]:
%sql
SELECT * FROM employee_delta

#### Insert statement

In [0]:
%sql
INSERT INTO employee_delta VALUES(1, 'Rohan', 'data engineer', 56, '2022-02-21', 60000, 3, 'Bangalore')

In [0]:
%sql
SELECT * FROM employee_delta

#### Alter table

In [0]:
%sql
ALTER TABLE employee_delta ADD COLUMN regisnation CHAR

#### Truncate, Delete, Drop a table

In [0]:
%sql
TRUNCATE TABLE testDB.testTable;
SELECT * FROM testDB.testTable;

#### Insert into a table

In [0]:
%sql
INSERT INTO testDB.employee (name) 
VALUES ("Simran")

In [0]:
df_test = spark.sql("SELECT * FROM testDB.testTable")
df_test.display()

In [0]:
df_employee = spark.sql(f"SELECT * FROM {employeeTable}")
df_employee.display()

In [0]:
names = ["Alice", "Bob", "Charlie", "David", "Emily"]

# Create a DataFrame with a column of names
df_names = spark.createDataFrame([(name,) for name in names], ["name"])
df_names.show()

In [0]:
df_names.write.option("mergeSchema", "true").mode("append").saveAsTable(employeeTable)

In [0]:
%sql
SELECT * FROM testDB.employee

In [0]:
for i in df_names.collect():
    schema = StructType([StructField("id", IntegerType(), True)\
                         ,StructField("name", StringType(), True)])
    df_temp = spark.createDataFrame(i.name, schema = schema)
    #print(i.name)
    df_temp.write.option("mergeSchema", "true").mode("append").saveAsTable(employeeTable)

In [0]:
df_test.truncate()

In [0]:
df_test.printSchema()

#### String functions in SQL

##### Locate()

In [0]:
%sql
SELECT LOCATE("a", "aabbbcccd")

In [0]:
%sql
SELECT LOCATE("b", "aabboioppbcccd",5)

In [0]:
%sql
SELECT LOCATE("z", "aabboioppbcccd")

#### DateTime in SQL

1. CURRENT_DATE
2. GETDATE()
3. CURRENT_TIMESTAMP
4. CONVERT(date, 'timestamp')
5. DATEADD(Hour/Minute/Second , value, GETDATE())
6. TIMESTAMPDIFF(Second/Minute/Hour, start_timestamp, end_timestamp) -> start_timestamp < end_timestamp
7. DATE_FORMAT(datecolumn, "yyyy-MM-dd")

##### current_timestamp

In [0]:
%sql
SELECT CURRENT_TIMESTAMP AS current_datetimestamp;

##### getdate()

In [0]:
%sql
SELECT GETDATE() AS current_timestamp_getdate

##### dateadd()

In [0]:
%sql
SELECT CURRENT_TIMESTAMP(), DATEADD(Hour, 5 ,GETDATE()) AS added_hours

In [0]:
%sql
SELECT CURRENT_TIMESTAMP(), DATEADD(Minute, 2, CURRENT_TIMESTAMP()) AS added_minutes

In [0]:
%sql
SELECT CURRENT_TIMESTAMP(), DATEADD(Second, 55, CURRENT_TIMESTAMP()) AS added_seconds_55

##### datediff()

In [0]:
%sql
SELECT DATEDIFF('2023-05-02','2023-04-26')

##### from_utc_timestamp

In [0]:
%sql
SELECT from_utc_timestamp(GETDATE(), 'Asia/Seoul') AS Seoul_Time;

In [0]:
%sql
SELECT from_utc_timestamp(GETDATE(), 'GMT+5:30') AS India_Current_Time;

##### timestampdiff

In [0]:
%sql
SELECT TIMESTAMPDIFF(Hour, CURRENT_TIMESTAMP, DATEADD(Hour, 1, CURRENT_TIMESTAMP)) AS time_difference

In [0]:
%sql
SELECT TIMESTAMPDIFF(Minute, CURRENT_TIMESTAMP, DATEADD(Hour, 1, CURRENT_TIMESTAMP)) AS time_difference

##### date_format()

In [0]:
%sql
SELECT CURRENT_DATE 

In [0]:
%sql
SELECT date_format(CURRENT_DATE, "yyyy/MM/dd") as  date_format

In [0]:
%sql
SELECT date_format(CURRENT_DATE, "yyyy-MM-dd") as  date_format

##### unix_timestamp()
This is defined as number of seconds passed since '1970-01-01 00:00:00' UTC

In [0]:
%sql
SELECT unix_timestamp('1970-01-01 00:00:00')

In [0]:
%sql
SELECT CURRENT_TIMESTAMP, UNIX_TIMESTAMP();

In [0]:
%sql
SELECT CURRENT_TIMESTAMP, UNIX_TIMESTAMP(DATEADD(Hour, 1 ,CURRENT_TIMESTAMP)); -- added 1 hour = 3600 seconds

## Delta tables

#### Paths and Variables for delta

In [0]:
display(dbutils.fs.ls('/user'))

In [0]:
path1 = '/user/delta_table/'
path2 = '/user/deltaTables/'
metastore = '/user/metastore/'

### creating temp view

In [0]:
df = spark.read.format("delta").load('/user/deltaTables/sample')

In [0]:
df.createTempView("sampleTable")

In [0]:
%sql
SELECT * FROM sampleTable

In [0]:
display(dbutils.fs.ls('/user'))

In [0]:
%sql
SHOW DATABASES

In [0]:
%sql
CREATE TABLE (
first_name STRING,
last_name STRING,
country STRING)
USING DELTA USING '/user/metastore/country/'

### Partitioned Delta Lake

In [0]:
df = spark.createDataFrame([('Ernesto', 'Guevara', 'Argentina'),
                           ('Maria', 'Sharapova', 'Russia'),
                           ('Bruce', 'Lee', 'China'),
                           ('Jack', 'Ma', 'China')], ['first_name', 'last_name', 'country'])
df.show()

In [0]:
df.repartition(col("country")).write.partitionBy("country").format("delta").saveAsTable("country_people")

In [0]:
%sql
SELECT * FROM country_people

In [0]:
%sql
DESCRIBE EXTENDED country_people

In [0]:
display(dbutils.fs.ls('dbfs:/user/hive/warehouse/country_people/country=China/'))